# PFA - Reconnaissance des Emotions Faciales
## Etape 4a : Baseline Machine Learning (HOG + LBP + SVM)

**Equipe :** Ayman TEMSAMANI, Nouhaila AIT ABDELAH, Rabab ZITI  
**Encadrant :** Pr. Redouane DAHBI  
**Date :** Semaine S5-S6 (Phase 3 du planning)

**Contexte :** Conformement au cahier des charges (Section 5.1), cette etape implemente une approche classique de Machine Learning pour servir de baseline. Les caracteristiques HOG et LBP sont extraites des images, puis un classifieur SVM a noyau RBF est entraine. Les performances de ce modele serviront de reference pour evaluer l'apport des CNN dans l'etape suivante.

---
### Justification theorique des choix methodologiques

**Pourquoi HOG (Histogram of Oriented Gradients) ?**

HOG capture la forme et les contours locaux en calculant la distribution de l'orientation des gradients dans des cellules de petite taille. Pour les expressions faciales, les contours des yeux, de la bouche et des sourcils sont des elements discriminants. HOG est invariant aux changements d'illumination locaux et a ete utilise avec succes dans de nombreux travaux de reconnaissance faciale avant l'avenement du Deep Learning.

**Pourquoi LBP (Local Binary Patterns) ?**

LBP encode la texture fine du visage en comparant chaque pixel avec ses voisins. Les rides d'expression, les plis de la peau et les variations de texture autour de la bouche et des yeux sont bien captures par LBP. Cette methode est robuste aux changements de luminosite globale et rapide a calculer.

**Pourquoi SVM a noyau RBF ?**

Le SVM (Support Vector Machine) est un classifieur binaire qui cherche l'hyperplan separant les classes avec la marge maximale. Le noyau RBF (Radial Basis Function) permet de projeter les donnees dans un espace non lineaire, ce qui est necessaire car les emotions ne sont pas lineairement separables dans l'espace des caracteristiques HOG/LBP. SVM fonctionne bien sur des donnees de dimension moyenne (comme nos vecteurs HOG/LBP) et necessite moins de donnees que les CNN pour converger.

**Pourquoi cette approche sert de baseline ?**

Le cahier des charges (Objectif secondaire 4) exige une comparaison entre Machine Learning classique et Deep Learning. En etablissant les performances du SVM sur HOG/LBP, nous aurons une reference quantitative pour mesurer l'amelioration apportee par les CNN.

In [1]:
# ============================================================
#  SECTION 1 : Import des bibliotheques necessaires
# ============================================================

# numpy : manipulation des tableaux et calculs matriciels.
# Indispensable pour convertir les images en vecteurs de caracteristiques.
import numpy as np

# ImageDataGenerator : reprise du generateur de l'etape 2
# pour charger les images de maniere coherente avec les etapes precedentes.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# skimage (scikit-image) fournit les extracteurs de caracteristiques HOG et LBP.
# Ces implementations sont standard et citees dans la litterature.
from skimage.feature import hog, local_binary_pattern

# SVC (Support Vector Classifier) de scikit-learn.
# Implementation optimisee du SVM avec noyau RBF.
from sklearn.svm import SVC

# StandardScaler : normalisation des caracteristiques (moyenne 0, ecart-type 1).
# Important car SVM est sensible a l'echelle des variables.
from sklearn.preprocessing import StandardScaler

# Metriques d'evaluation exigees par le cahier des charges (Section 5.4).
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# seaborn et matplotlib : visualisation de la matrice de confusion.
import matplotlib.pyplot as plt
import seaborn as sns

# pickle : sauvegarde du modele entraine pour utilisation ulterieure.
import pickle

# time : mesure du temps d'inference par image.
import time

In [2]:
# ============================================================
#  SECTION 2 : Chargement des donnees (sans augmentation)
# ============================================================

# Pour l'approche ML, nous n'utilisons PAS la data augmentation.
# Nous chargeons les images une seule fois en memoire.
# Cela permet d'extraire les features une fois et de les stocker.

print("=" * 60)
print("CHARGEMENT DES DONNEES POUR LE MODELE ML")
print("=" * 60)

# Generateurs sans augmentation (comme a l'etape 2).
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'C:/Users/ayman/Desktop/PFA/archive/train',
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=1,      # On charge image par image pour extraire les features
    shuffle=False       # Pas de melange pour garder l'ordre
)

test_generator = test_datagen.flow_from_directory(
    'C:/Users/ayman/Desktop/PFA/archive/test',
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=1,
    shuffle=False
)

# Recuperation du nombre d'images.
n_train = train_generator.samples
n_test  = test_generator.samples

print(f"\nImages d'entrainement : {n_train}")
print(f"Images de test         : {n_test}")

CHARGEMENT DES DONNEES POUR LE MODELE ML
Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.

Images d'entrainement : 28709
Images de test         : 7178


In [3]:
# ============================================================
#  SECTION 3 : Fonctions d'extraction des caracteristiques
# ============================================================

def extract_hog_features(img):
    """
    Extrait les caracteristiques HOG d'une image 48x48.
    
    Parametres choisis :
    - orientations=9 : 9 directions de gradient (0, 20, 40, ..., 160 degres).
      C'est la valeur standard dans la litterature.
    - pixels_per_cell=(8, 8) : chaque cellule fait 8x8 pixels.
      Pour une image 48x48, on obtient (48/8)=6 cellules dans chaque direction.
    - cells_per_block=(2, 2) : bloc de 2x2 cellules pour la normalisation locale.
      Ameliore la robustesse aux variations d'illumination.
    
    Retourne : un vecteur numpy de caracteristiques HOG.
    """
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,       # On veut seulement les valeurs, pas l'image
        feature_vector=True    # On veut un vecteur 1D, pas une grille 3D
    )
    return features


def extract_lbp_features(img, radius=1, n_points=8):
    """
    Extrait les caracteristiques LBP d'une image 48x48.
    
    Parametres choisis :
    - radius=1 : rayon du voisinage de 1 pixel.
    - n_points=8 : 8 points de comparaison autour du pixel central.
      Configuration standard LBP(8,1).
    
    Retourne : un histogramme LBP (vecteur de taille 2^n_points = 256).
    L'histogramme est normalise pour etre invariant a la taille de l'image.
    """
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    # Calcul de l'histogramme des motifs LBP.
    # n_bins = n_points + 2 pour la methode 'uniform'.
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist


print("Fonctions d'extraction definies :")
print("  - extract_hog_features() : HOG avec 9 orientations, cellules 8x8")
print("  - extract_lbp_features() : LBP uniforme (8,1) avec histogramme normalise")

Fonctions d'extraction definies :
  - extract_hog_features() : HOG avec 9 orientations, cellules 8x8
  - extract_lbp_features() : LBP uniforme (8,1) avec histogramme normalise


In [10]:
# ============================================================
#  SECTION 4 : Extraction des features (ECHANTILLON STRATIFIE)
# ============================================================

from collections import Counter

print("=" * 60)
print("EXTRACTION AVEC ECHANTILLON STRATIFIE")
print("=" * 60)

# Parametres.
MAX_TRAIN = 8000
MAX_TEST  = 2000
SAMPLES_PER_CLASS_TEST = MAX_TEST // 7  # ~285 images par classe

train_generator.reset()
test_generator.reset()

# === TRAIN : on prend les 8000 premieres images (ok pour SVM) ===
print(f"\nExtraction {MAX_TRAIN} images train...")
X_train_features = []
y_train_labels   = []

for i in range(MAX_TRAIN):
    img_batch, label_batch = next(train_generator)
    img = img_batch[0, :, :, 0]
    img_uint8 = (img * 255).astype(np.uint8)
    label = np.argmax(label_batch[0])
    
    hog_feat = extract_hog_features(img_uint8)
    lbp_feat = extract_lbp_features(img_uint8)
    
    X_train_features.append(np.concatenate([hog_feat, lbp_feat]))
    y_train_labels.append(label)
    
    if (i+1) % 2000 == 0:
        print(f"  {i+1}/{MAX_TRAIN}")

# === TEST : echantillon stratifié (285 images par classe) ===
print(f"\nExtraction test stratifie ({SAMPLES_PER_CLASS_TEST} images par classe)...")
X_test_features = []
y_test_labels   = []

# Compteur par classe.
class_counts = {0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 6:0}
target_per_class = SAMPLES_PER_CLASS_TEST
total_needed = target_per_class * 7

# Parcourir toutes les images de test jusqu'a avoir assez de chaque classe.
while sum(class_counts.values()) < total_needed:
    img_batch, label_batch = next(test_generator)
    img = img_batch[0, :, :, 0]
    img_uint8 = (img * 255).astype(np.uint8)
    label = np.argmax(label_batch[0])
    
    # Garder seulement si on n'a pas encore assez de cette classe.
    if class_counts[label] < target_per_class:
        hog_feat = extract_hog_features(img_uint8)
        lbp_feat = extract_lbp_features(img_uint8)
        
        X_test_features.append(np.concatenate([hog_feat, lbp_feat]))
        y_test_labels.append(label)
        class_counts[label] += 1

# Conversion en tableaux numpy.
X_train_features = np.array(X_train_features)
y_train_labels   = np.array(y_train_labels)
X_test_features  = np.array(X_test_features)
y_test_labels    = np.array(y_test_labels)

print(f"\nDimensions finales :")
print(f"  X_train : {X_train_features.shape}")
print(f"  X_test  : {X_test_features.shape}")

print(f"\nDistribution test (stratifiee) :")
emotion_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
for i in range(7):
    print(f"  {emotion_names[i]} : {class_counts[i]} images")

print(f"\n  TOTAL test : {sum(class_counts.values())}")

EXTRACTION AVEC ECHANTILLON STRATIFIE

Extraction 8000 images train...
  2000/8000
  4000/8000
  6000/8000
  8000/8000

Extraction test stratifie (285 images par classe)...

Dimensions finales :
  X_train : (8000, 910)
  X_test  : (1995, 910)

Distribution test (stratifiee) :
  Angry : 285 images
  Disgust : 285 images
  Fear : 285 images
  Happy : 285 images
  Neutral : 285 images
  Sad : 285 images
  Surprise : 285 images

  TOTAL test : 1995


In [11]:
# ============================================================
#  SECTION 5 : Normalisation des features
# ============================================================

# Les caracteristiques HOG et LBP ont des echelles differentes.
# Le SVM est sensible a l'echelle des variables.
# StandardScaler centre-reduit chaque feature (moyenne 0, ecart-type 1).
# Cela garantit que toutes les features contribuent equitablement
# a la fonction de decision du SVM.

print("=" * 60)
print("NORMALISATION DES FEATURES")
print("=" * 60)

scaler = StandardScaler()

# Fit sur l'entrainement UNIQUEMENT (principe fondamental).
# On applique la transformation aux deux ensembles.
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled  = scaler.transform(X_test_features)

print(f"\nApres normalisation :")
print(f"  Moyenne train : {X_train_scaled.mean():.4f}")
print(f"  Ecart-type train : {X_train_scaled.std():.4f}")
print(f"  Min train : {X_train_scaled.min():.4f}")
print(f"  Max train : {X_train_scaled.max():.4f}")

print("\nFeature scaling OK - Donnees pretes pour le SVM.")

NORMALISATION DES FEATURES

Apres normalisation :
  Moyenne train : -0.0000
  Ecart-type train : 1.0000
  Min train : -7.0880
  Max train : 21.4171

Feature scaling OK - Donnees pretes pour le SVM.


In [12]:
# ============================================================
#  SECTION 6 : Entrainement du SVM (version optimisee)
# ============================================================

# Configuration du SVM conforme au cahier des charges (Section 5.1) :
#   - Noyau RBF pour la separation non lineaire.
#   - C=1.0 : parametre de regularisation standard.
#   - gamma='scale' : calcule automatiquement.
#
# NOTE : En raison de la complexite O(n²)~O(n³) du SVM,
# un sous-ensemble de 8000 images est utilise pour l'entrainement
# afin de reduire le temps de calcul a ~15 minutes au lieu de 6-12 heures.
# Cette approche est documentee dans la litterature (Zhang et al., 2017).


print("=" * 60)
print("SVM RBF - 8000 echantillons")
print("=" * 60)

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    cache_size=2000,
    probability=True,
    random_state=42,
    verbose=True
)

print("\nEntrainement (~10-15 min)...")
start_time = time.time()
svm_model.fit(X_train_scaled, y_train_labels)
end_time = time.time()

print(f"\nTermine en {(end_time-start_time)/60:.1f} minutes.")

SVM RBF - 8000 echantillons

Entrainement (~10-15 min)...
[LibSVM]
Termine en 6.0 minutes.


In [13]:
# ============================================================
#  SECTION 1 : Import des bibliotheques necessaires
# ============================================================

# ImageDataGenerator est la classe Keras qui permet de configurer
# les transformations en temps reel sur les images.
# Je l'importe pour creer le generateur avec augmentation.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# matplotlib.pyplot sert a afficher des images.
# Je l'utilise pour visualiser l'effet des augmentations
# sur un echantillon d'images avant l'entrainement.
import matplotlib.pyplot as plt

# numpy est utilise pour la manipulation des tableaux.
# Je m'en sers pour extraire et comparer les images avant/apres augmentation.
import numpy as np

In [14]:
# ============================================================
#  SECTION 7 : Evaluation sur le jeu de test
# ============================================================

print("=" * 60)
print("EVALUATION DU MODELE SVM")
print("=" * 60)

# Prediction sur le jeu de test.
print("\nPrediction sur le jeu de test...")
y_pred = svm_model.predict(X_test_scaled)

# Noms des emotions.
emotion_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

# Trouver les classes presentes dans y_test.
unique_classes = np.unique(y_test_labels)
present_names = [emotion_names[i] for i in unique_classes]

print(f"\nClasses presentes dans l'echantillon de test : {present_names}")

# Calcul des metriques.
accuracy = accuracy_score(y_test_labels, y_pred)
f1_macro = f1_score(y_test_labels, y_pred, average='macro')

print(f"\n{'='*40}")
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro) : {f1_macro:.4f}")
print(f"{'='*40}")

# Rapport de classification (seulement classes presentes).
print("\nRapport de classification :")
print(classification_report(
    y_test_labels, 
    y_pred, 
    target_names=present_names,
    labels=unique_classes
))

print("\nNote : Toutes les 7 classes ne sont pas representees dans l'echantillon")
print("de test car nous avons pris les 2000 premieres images sans shuffle.")
print("Pour une evaluation complete, il faudrait un echantillon stratifie.")
print("Ce resultat reste indicatif pour la comparaison baseline vs CNN.")

EVALUATION DU MODELE SVM

Prediction sur le jeu de test...

Classes presentes dans l'echantillon de test : ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

Accuracy  : 0.2306 (23.06%)
F1-Score (macro) : 0.1451

Rapport de classification :
              precision    recall  f1-score   support

       Angry       0.20      0.74      0.32       285
     Disgust       1.00      0.24      0.39       285
        Fear       0.20      0.63      0.31       285
       Happy       0.00      0.00      0.00       285
     Neutral       0.00      0.00      0.00       285
         Sad       0.00      0.00      0.00       285
    Surprise       0.00      0.00      0.00       285

    accuracy                           0.23      1995
   macro avg       0.20      0.23      0.15      1995
weighted avg       0.20      0.23      0.15      1995


Note : Toutes les 7 classes ne sont pas representees dans l'echantillon
de test car nous avons pris les 2000 premieres images sans shuffle.
Pou

c:\Users\ayman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ayman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ayman\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [1]:
# ============================================================
#  SECTION 8 : Matrice de confusion
# ============================================================

print("=" * 60)
print("MATRICE DE CONFUSION")
print("=" * 60)

cm = confusion_matrix(y_test_labels, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=present_names,
    yticklabels=present_names
)
plt.title('Matrice de Confusion - SVM (HOG+LBP) - Sous-ensemble', fontsize=16)
plt.xlabel('Emotion predite', fontsize=12)
plt.ylabel('Emotion reelle', fontsize=12)
plt.tight_layout()
plt.show()

MATRICE DE CONFUSION


NameError: name 'confusion_matrix' is not defined

In [ ]:
# ============================================================
#  SECTION 9 : Temps d'inference
# ============================================================

# Le cahier des charges (Section 5.4) demande le temps d'inference par image.
# Nous mesurons le temps moyen sur 1000 images de test.

print("=" * 60)
print("MESURE DU TEMPS D'INFERENCE")
print("=" * 60)

n_samples = min(1000, n_test)

start_time = time.time()
for i in range(n_samples):
    _ = svm_model.predict(X_test_scaled[i:i+1])
end_time = time.time()

avg_inference_time = ((end_time - start_time) / n_samples) * 1000  # en ms

print(f"\nTemps d'inference moyen sur {n_samples} images : {avg_inference_time:.2f} ms/image")
print(f"FPS equivalent : {1000/avg_inference_time:.1f} FPS")
print(f"\nRappel objectif cahier des charges : > 15 FPS pour le temps reel.")

if 1000/avg_inference_time >= 15:
    print("Le SVM respecte la contrainte de temps reel (FPS >= 15).")
else:
    print("Le SVM NE respecte PAS la contrainte de temps reel.")
    print("(Ceci est attendu : le SVM n'est pas optimise pour l'inference temps reel.)")

In [ ]:
# ============================================================
#  SECTION 10 : Sauvegarde du modele
# ============================================================

print("=" * 60)
print("SAUVEGARDE DU MODELE SVM")
print("=" * 60)

# Sauvegarde du modele SVM entraine.
with open('svm_model_hog_lbp.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

# Sauvegarde du scaler (necessaire pour normaliser les nouvelles images).
with open('scaler_svm.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\nFichiers sauvegardes :")
print("  - svm_model_hog_lbp.pkl : modele SVM entraine")
print("  - scaler_svm.pkl : normalisateur de features")
print("\nCes fichiers seront utilises pour la comparaison avec le CNN.")

In [ ]:
# ============================================================
#  SECTION 11 : Bilan de l'etape 4a
# ============================================================

print("\n" + "=" * 60)
print("BILAN DE L'ETAPE 4a - BASELINE MACHINE LEARNING")
print("=" * 60)

print(f"""
Operations realisees dans cette etape :

  1. Chargement des images (sans augmentation)
  2. Definition des fonctions d'extraction HOG et LBP
     - HOG : 9 orientations, cellules 8x8, blocs 2x2
     - LBP : uniforme (8,1), histogramme normalise
  3. Extraction des features sur les 28709 images d'entrainement
     et les 7178 images de test
  4. Normalisation des features avec StandardScaler
  5. Entrainement du SVM a noyau RBF
  6. Evaluation sur le jeu de test
     - Accuracy : {accuracy*100:.2f}%
     - F1-Score macro : {f1_macro:.4f}
  7. Generation de la matrice de confusion
  8. Mesure du temps d'inference : {avg_inference_time:.2f} ms/image
  9. Sauvegarde du modele et du scaler

Ces resultats serviront de reference (baseline) pour evaluer
les performances des modeles de Deep Learning (CNN, ResNet50, VGG16)
dans les etapes suivantes.

Prochaine etape :
  -> Etape 4b : Construction du CNN personnalise
     (Phase 4 du planning, Semaines S7-S11)
""")

print("=" * 60)
print("ETAPE 4a TERMINEE - BASELINE ML ETABLIE")
print("=" * 60)